# CNN 3D v3: MoViNet with Validation-Tuned Left-Turn Decisions

This notebook classifies `straight`, `right-turn`, and `left-turn` using
short RGB clips that contain **only frames before the labeled target
frame**. It preserves the exact train/validation/test rows and ordering
created by `CNN_2D_resplit.ipynb` and used by `CNN_3D.ipynb`.

The video backbone is the CPU-friendly streaming MoViNet-A0 variant
pretrained on Kinetics-600. Because this
project is being run on a CPU-only laptop, each clip is passed through the
frozen backbone **once** and the resulting embedding is cached. All
hyperparameter tuning then trains small dense classification heads on the
cached embeddings. This keeps genuine video transfer learning while
avoiding repeated, prohibitively slow MoViNet passes.

Version 3 adds one validation-only decision hyperparameter: a multiplier
applied to the `left-turn` probability before selecting the predicted
class. The multiplier is tuned only on the train-internal validation fold,
locked before final training, and then applied unchanged to the shared
validation and held-out test folds.

The shared validation fold is used only for the final report. Model
selection and early stopping use a stratified validation subset taken from
the official training fold, and the test fold is evaluated once at the end.


## Why this design is appropriate

The strongest classical benchmark indicates that temporal motion is more
informative than a single target image. MoViNet supplies motion-aware
Kinetics features learned from a large video dataset, while the custom head
learns the three Waymo maneuver classes.

A full MoViNet forward pass is still expensive on a CPU. Repeating that pass
during every epoch and tuning trial would make the notebook impractical.
The frozen-embedding design therefore separates the work:

1. construct the same prior-only clips for every split;
2. run each clip through frozen Kinetics-pretrained MoViNet exactly once;
3. cache one video embedding per clip;
4. tune the custom dense head and left-turn decision multiplier on a
   train-internal validation fold;
5. retrain the selected head on all official training rows;
6. report shared-validation and held-out-test metrics;
7. save `CNN_3D_v3.npy` and `labels_CNN_3D_v3.npy`.

Horizontal flipping is intentionally absent because it would exchange the
semantic meanings of left and right turns. Pixel augmentation is also not
applied during cached-head tuning: after a frozen embedding is created,
pixel-level augmentation would require another expensive backbone pass.

Implementation references: TensorFlow's [MoViNet transfer-learning
tutorial](https://www.tensorflow.org/tutorials/video/transfer_learning_with_movinet)
and the official [Model Garden MoViNet
implementation](https://github.com/tensorflow/models/tree/master/official/projects/movinet).


## One-time environment requirement

Run this notebook from VS Code connected to **WSL2/Linux** and select the
`Python (waymo-movinet)` kernel. From a WSL terminal opened at the repository
root:

```bash
python3.12 -m venv ~/.venvs/waymo-movinet
source ~/.venvs/waymo-movinet/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install \
  "tensorflow==2.20.0" "tf-keras==2.20.1" \
  "tensorflow-text==2.20.1" "tf-models-official==2.20.0" \
  pandas matplotlib scikit-learn imageio ipykernel
python -m ipykernel install --user \
  --name waymo-movinet \
  --display-name "Python (waymo-movinet)"
```

Then choose `Python (waymo-movinet)` in the notebook kernel picker and use
**Restart Kernel and Run All Cells**.

This version deliberately runs TensorFlow on the CPU. The first complete
execution is the long one because it builds the clip cache and extracts
MoViNet embeddings. Each split is saved separately, so later executions
reuse completed clip and embedding caches.


# Step 1: Import packages and set reproducible behavior


In [ ]:
import os

# This laptop has an AMD integrated GPU, which TensorFlow cannot use through
# CUDA. Hiding CUDA before TensorFlow is imported prevents repeated CUDA
# initialization warnings. Set USE_GPU=True only in a compatible NVIDIA
# CUDA environment.
USE_GPU = False
if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import gc
import hashlib
import json
import math
import pickle
import random
import re
import tarfile
import time
from contextlib import contextmanager
from pathlib import Path, PureWindowsPath

import imageio.v2 as imageio

# Force VS Code/Jupyter to render Matplotlib figures in cell outputs.
try:
    get_ipython().run_line_magic(
        'matplotlib',
        'inline',
    )
except NameError:
    pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

try:
    import tf_keras as keras
    from official.projects.movinet.modeling import movinet
    from official.projects.movinet.modeling import movinet_model
except ImportError as error:
    raise ImportError(
        'MoViNet dependencies are missing or incompatible. Use the pinned '
        'Python (waymo-movinet) WSL2/Linux environment described in the '
        'One-time environment requirement above, restart the kernel, and '
        'rerun this notebook.'
    ) from error

from IPython.display import Image as IPythonImage
from IPython.display import Markdown, display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print('TensorFlow deterministic operations enabled.')
except Exception as error:
    print(
        'TensorFlow deterministic operations were not '
        f'enabled: {error}'
    )

print(f'TensorFlow version: {tf.__version__}')
print(
    'Legacy Keras version: '
    f'{getattr(keras, "__version__", "unknown")}'
)
print(f'NumPy version: {np.__version__}')
print(
    'Execution device: '
    f'{"GPU" if tf.config.list_physical_devices("GPU") else "CPU"}'
)


# Step 2: Define project paths, clip settings, and cache locations

The split CSVs remain the source of truth. Version 3 intentionally reuses
the unchanged video clips, MoViNet embeddings, GIFs, and pretrained
checkpoint already stored beneath
`data/processed/waymo_e2e/CNN_3D_v2/`. With both overwrite flags set to
`False`, no expensive clip construction or MoViNet extraction is repeated.


In [ ]:
# Use the project root whether execution starts from the project
# root or from inside the notebooks directory.
current_dir = Path.cwd()
PROJECT_ROOT = (
    current_dir.parent
    if current_dir.name == 'notebooks'
    else current_dir
)

CNN_2D_DATA_DIR = (
    PROJECT_ROOT
    / 'data'
    / 'processed'
    / 'waymo_e2e'
    / 'CNN_2D'
)
CNN_3D_V2_DATA_DIR = (
    PROJECT_ROOT
    / 'data'
    / 'processed'
    / 'waymo_e2e'
    / 'CNN_3D_v2'
)
CLIP_CACHE_DIR = (
    CNN_3D_V2_DATA_DIR / 'clip_cache'
)
EMBEDDING_CACHE_DIR = (
    CNN_3D_V2_DATA_DIR / 'embedding_cache'
)
GIF_OUTPUT_DIR = (
    CNN_3D_V2_DATA_DIR / 'example_gifs'
)
PRETRAINED_MODEL_DIR = (
    CNN_3D_V2_DATA_DIR / 'pretrained'
)

for output_directory in [
    CNN_3D_V2_DATA_DIR,
    CLIP_CACHE_DIR,
    EMBEDDING_CACHE_DIR,
    GIF_OUTPUT_DIR,
    PRETRAINED_MODEL_DIR,
]:
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

SPLIT_NAMES = ['train', 'val', 'test']

DF_PATHS = {
    split_name: (
        CNN_2D_DATA_DIR / f'df_{split_name}.csv'
    )
    for split_name in SPLIT_NAMES
}
Y_PATHS = {
    split_name: (
        CNN_2D_DATA_DIR / f'y_{split_name}.csv'
    )
    for split_name in SPLIT_NAMES
}
CLIP_CACHE_PATHS = {
    split_name: (
        CLIP_CACHE_DIR
        / f'X_{split_name}_clips.npy'
    )
    for split_name in SPLIT_NAMES
}
CLIP_METADATA_PATHS = {
    split_name: (
        CLIP_CACHE_DIR
        / f'clip_metadata_{split_name}.csv'
    )
    for split_name in SPLIT_NAMES
}
CACHE_CONFIG_PATH = (
    CLIP_CACHE_DIR / 'clip_cache_config.json'
)
EMBEDDING_CACHE_PATHS = {
    split_name: (
        EMBEDDING_CACHE_DIR
        / f'movinet_a0_stream_{split_name}_embeddings.npy'
    )
    for split_name in SPLIT_NAMES
}
EMBEDDING_CONFIG_PATH = (
    EMBEDDING_CACHE_DIR
    / 'embedding_cache_config.json'
)

MIN_DESIRED_PRIOR_FRAMES = 12
CLIP_LENGTH = 16
MAX_LOOKBACK_FRAMES = 20

IMAGE_HEIGHT = 96
IMAGE_WIDTH = 256
IMAGE_CHANNELS = 3
INPUT_SHAPE = (
    CLIP_LENGTH,
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    IMAGE_CHANNELS,
)

# Rebuild only after changing the corresponding settings.
OVERWRITE_CLIP_CACHE = False
OVERWRITE_EMBEDDING_CACHE = False

# A held-out subset of the official training fold is used for
# early stopping and hyperparameter selection.
INTERNAL_VALIDATION_FRACTION = 0.15

# Clip-consistent augmentation is available for inspection, but cached
# frozen embeddings are extracted from unaugmented clips.
MAX_BRIGHTNESS_DELTA = 0.06
MAX_CONTRAST_DELTA = 0.12

MOVINET_MODEL_ID = 'a0'
MOVINET_VARIANT = 'stream'
MOVINET_CHECKPOINT_FILENAME = (
    f'movinet_{MOVINET_MODEL_ID}_'
    f'{MOVINET_VARIANT}.tar.gz'
)
MOVINET_CHECKPOINT_URL = (
    'https://storage.googleapis.com/'
    'tf_model_garden/vision/movinet/'
    + MOVINET_CHECKPOINT_FILENAME
)

if not (
    MIN_DESIRED_PRIOR_FRAMES
    <= CLIP_LENGTH
    <= MAX_LOOKBACK_FRAMES
):
    raise ValueError(
        'CLIP_LENGTH must be between '
        f'{MIN_DESIRED_PRIOR_FRAMES} and '
        f'{MAX_LOOKBACK_FRAMES}.'
    )

for required_path in (
    list(DF_PATHS.values())
    + list(Y_PATHS.values())
):
    if not required_path.exists():
        raise FileNotFoundError(
            f'Required split file not found: '
            f'{required_path}'
        )

print(f'Project root: {PROJECT_ROOT}')
print(
    'Transfer-learning data directory: '
    f'{CNN_3D_V2_DATA_DIR}'
)
print(f'MoViNet input shape: {INPUT_SHAPE}')
print(
    f'Prior-frame policy: sample {CLIP_LENGTH} '
    f'frames from the most recent '
    f'{MAX_LOOKBACK_FRAMES} eligible frames'
)
print(
    'Embedding cache: '
    f'{EMBEDDING_CACHE_DIR}'
)


# Step 3: Load and verify the unchanged train, validation, and test splits

The dataframe and label CSV for each split must have equal row counts
and identical label ordering. Fingerprints are printed so this run can
be tied to the exact same split files used by the other benchmarks.


In [ ]:
split_dfs = {}
split_label_dfs = {}
split_summary_rows = []
split_fingerprints = {}

for split_name in SPLIT_NAMES:
    split_df = pd.read_csv(
        DF_PATHS[split_name]
    )
    label_df = pd.read_csv(
        Y_PATHS[split_name]
    )

    required_columns = {
        'target_frame_path',
        'class_label',
    }
    if not required_columns.issubset(
        split_df.columns
    ):
        raise ValueError(
            f'{DF_PATHS[split_name].name} must '
            f'contain {sorted(required_columns)}.'
        )

    if list(label_df.columns) != [
        'class_label'
    ]:
        raise ValueError(
            f'{Y_PATHS[split_name].name} must '
            'contain only class_label.'
        )

    if len(split_df) != len(label_df):
        raise ValueError(
            f'Row mismatch for {split_name}: '
            f'{len(split_df)} dataframe rows and '
            f'{len(label_df)} labels.'
        )

    dataframe_labels = (
        split_df['class_label']
        .reset_index(drop=True)
    )
    separate_labels = (
        label_df['class_label']
        .reset_index(drop=True)
    )
    if not dataframe_labels.equals(
        separate_labels
    ):
        raise ValueError(
            f'Label order mismatch in '
            f'{split_name}.'
        )

    split_dfs[split_name] = split_df
    split_label_dfs[split_name] = label_df

    fingerprint_text = (
        split_df[[
            'target_frame_path',
            'class_label',
        ]]
        .to_csv(index=False)
        .encode('utf-8')
    )
    split_fingerprints[split_name] = (
        hashlib.sha256(
            fingerprint_text
        ).hexdigest()
    )

    class_counts = (
        split_df['class_label']
        .value_counts()
    )
    split_summary_rows.append({
        'split': split_name,
        'sequences': len(split_df),
        'straight': int(
            class_counts.get('straight', 0)
        ),
        'right-turn': int(
            class_counts.get(
                'right-turn',
                0,
            )
        ),
        'left-turn': int(
            class_counts.get(
                'left-turn',
                0,
            )
        ),
        'fingerprint': (
            split_fingerprints[
                split_name
            ][:12]
        ),
    })

display(pd.DataFrame(split_summary_rows))

CLASS_NAMES = [
    'straight',
    'right-turn',
    'left-turn',
]
CLASS_TO_ID = {
    class_name: class_id
    for class_id, class_name
    in enumerate(CLASS_NAMES)
}
ID_TO_CLASS = {
    class_id: class_name
    for class_name, class_id
    in CLASS_TO_ID.items()
}
NUM_CLASSES = len(CLASS_NAMES)

y_by_split = {}

for split_name in SPLIT_NAMES:
    encoded_labels = (
        split_label_dfs[
            split_name
        ]['class_label']
        .map(CLASS_TO_ID)
    )

    if encoded_labels.isna().any():
        unknown_labels = (
            split_label_dfs[
                split_name
            ]
            .loc[
                encoded_labels.isna(),
                'class_label',
            ]
            .unique()
        )
        raise ValueError(
            f'Unknown labels in {split_name}: '
            f'{unknown_labels}'
        )

    y_by_split[split_name] = (
        encoded_labels.to_numpy(
            dtype=np.int32
        )
    )

y_train = y_by_split['train']
y_val = y_by_split['val']
y_test = y_by_split['test']

print(
    f'Total sequences: '
    f'{sum(len(values) for values in y_by_split.values()):,}'
)


# Step 4: Create a train-internal validation fold

The official validation fold is **not** used for CNN early stopping or
transfer-learning hyperparameter selection. Instead, a reproducible,
stratified 85/15 split is made inside the official training fold.

After choosing the training plan, the model is reinitialized and trained
on every official training example for the selected number of epochs.
This leaves the shared validation fold clean for later feature-fusion
work.


In [ ]:
all_training_indices = np.arange(
    len(y_train)
)

(
    internal_training_indices,
    internal_validation_indices,
) = train_test_split(
    all_training_indices,
    test_size=INTERNAL_VALIDATION_FRACTION,
    random_state=SEED,
    shuffle=True,
    stratify=y_train,
)

# Sorting makes validation predictions align with y labels.
internal_training_indices = np.sort(
    internal_training_indices
)
internal_validation_indices = np.sort(
    internal_validation_indices
)

# Keep labels in the exact same order as the corresponding embedding
# rows selected by the two index arrays.
y_internal_training = y_train[
    internal_training_indices
]
y_internal_validation = y_train[
    internal_validation_indices
]


def balanced_class_weights(labels):
    labels = np.asarray(
        labels,
        dtype=np.int32,
    )
    class_counts = np.bincount(
        labels,
        minlength=NUM_CLASSES,
    )

    if np.any(class_counts == 0):
        raise ValueError(
            'Every class must be represented '
            'before computing class weights.'
        )

    weights = (
        len(labels)
        / (NUM_CLASSES * class_counts)
    )
    return {
        class_id: float(weights[class_id])
        for class_id in range(NUM_CLASSES)
    }


internal_class_weights = (
    balanced_class_weights(
        y_internal_training
    )
)
full_training_class_weights = (
    balanced_class_weights(y_train)
)

internal_summary_rows = []
for partition_name, indices in {
    'internal-train': (
        internal_training_indices
    ),
    'internal-validation': (
        internal_validation_indices
    ),
}.items():
    labels = y_train[indices]
    counts = np.bincount(
        labels,
        minlength=NUM_CLASSES,
    )
    internal_summary_rows.append({
        'partition': partition_name,
        'sequences': len(indices),
        **{
            class_name: int(
                counts[class_id]
            )
            for class_id, class_name
            in enumerate(CLASS_NAMES)
        },
    })

display(
    pd.DataFrame(internal_summary_rows)
)
print(
    'Internal-training class weights:'
)
print(internal_class_weights)

if set(internal_training_indices).intersection(
    set(internal_validation_indices)
):
    raise AssertionError(
        'Internal train/validation overlap.'
    )


# Step 5: Inventory numeric frame files and select prior-only clips

Only sequences referenced by the three split CSVs are indexed. The scan
uses `os.scandir()` and stores paths directly instead of calling
`Path.resolve()` on every `.npz` file. This is especially important when
the repository is located on the Windows-mounted `/mnt/c` drive in WSL2.

Progress is printed every 25,000 directory entries so a large directory
does not appear frozen.


In [ ]:
FRAME_NAME_PATTERN = re.compile(
    r'^(?P<sequence_id>.+)_'
    r'(?P<frame_index>\d+)\.npz$'
)


def resolve_frame_path(path_value):
    # Support project-relative paths, Linux paths, and absolute Windows
    # paths written into CSVs before the notebook moved into WSL2.
    path_text = str(path_value).strip()

    if re.match(r'^[A-Za-z]:[\\/]', path_text):
        windows_path = PureWindowsPath(path_text)
        drive_letter = windows_path.drive[0].lower()
        return (
            Path('/mnt')
            / drive_letter
            / Path(*windows_path.parts[1:])
        )

    frame_path = Path(
        path_text.replace('\\', '/')
    )
    if frame_path.is_absolute():
        return frame_path
    return (
        PROJECT_ROOT / frame_path
    )


def parse_frame_name(frame_path):
    match = FRAME_NAME_PATTERN.fullmatch(
        Path(frame_path).name
    )
    if match is None:
        return None

    return (
        match.group('sequence_id'),
        int(match.group('frame_index')),
    )


def build_frame_inventory(dataframes):
    # Determine which sequence IDs are actually needed in each directory.
    required_sequences_by_directory = {}

    for split_df in dataframes.values():
        for path_value in (
            split_df['target_frame_path']
        ):
            target_path = resolve_frame_path(
                path_value
            )
            parsed_target = parse_frame_name(
                target_path
            )

            if parsed_target is None:
                raise ValueError(
                    'Target filename does not end in a '
                    'numeric frame index: '
                    f'{target_path.name}'
                )

            sequence_id, _ = parsed_target
            required_sequences_by_directory.setdefault(
                target_path.parent,
                set(),
            ).add(sequence_id)

    inventory_by_index = {}
    total_scanned_entries = 0
    total_indexed_files = 0
    total_skipped_names = 0

    for (
        target_directory,
        required_sequence_ids,
    ) in sorted(
        required_sequences_by_directory.items(),
        key=lambda item: str(item[0]),
    ):
        if not target_directory.exists():
            raise FileNotFoundError(
                'Frame directory not found: '
                f'{target_directory}'
            )

        print(
            f'Scanning {target_directory} '
            f'for {len(required_sequence_ids):,} '
            'required sequences...'
        )
        directory_start_time = (
            time.perf_counter()
        )
        directory_entry_count = 0
        directory_indexed_count = 0

        # os.scandir returns names without performing a separate
        # resolve/stat filesystem request for every file.
        with os.scandir(
            target_directory
        ) as directory_entries:
            for directory_entry in (
                directory_entries
            ):
                directory_entry_count += 1
                total_scanned_entries += 1

                if (
                    directory_entry_count
                    % 25_000
                    == 0
                ):
                    print(
                        f'  scanned '
                        f'{directory_entry_count:,} '
                        'directory entries...'
                    )

                file_name = (
                    directory_entry.name
                )

                if not file_name.lower().endswith(
                    '.npz'
                ):
                    continue

                parsed_name = parse_frame_name(
                    file_name
                )
                if parsed_name is None:
                    total_skipped_names += 1
                    continue

                sequence_id, frame_index = (
                    parsed_name
                )
                if sequence_id not in (
                    required_sequence_ids
                ):
                    continue

                inventory_key = (
                    str(target_directory),
                    sequence_id,
                )
                inventory_by_index.setdefault(
                    inventory_key,
                    [],
                ).append((
                    frame_index,
                    target_directory
                    / file_name,
                ))

                directory_indexed_count += 1
                total_indexed_files += 1

        elapsed_seconds = (
            time.perf_counter()
            - directory_start_time
        )
        print(
            f'  completed '
            f'{directory_entry_count:,} entries; '
            f'indexed '
            f'{directory_indexed_count:,} files '
            f'in {elapsed_seconds:.1f} seconds'
        )

    frame_inventory = {
        inventory_key: sorted(
            frame_list,
            key=lambda item: item[0],
        )
        for inventory_key, frame_list
        in inventory_by_index.items()
    }

    expected_inventory_keys = {
        (
            str(target_directory),
            sequence_id,
        )
        for (
            target_directory,
            required_sequence_ids,
        ) in (
            required_sequences_by_directory
            .items()
        )
        for sequence_id in (
            required_sequence_ids
        )
    }
    missing_inventory_keys = (
        expected_inventory_keys
        - set(frame_inventory)
    )

    if missing_inventory_keys:
        missing_examples = sorted(
            missing_inventory_keys
        )[:5]
        raise KeyError(
            'No numeric frame files were found '
            f'for {len(missing_inventory_keys)} '
            'required sequences. Examples: '
            f'{missing_examples}'
        )

    print(
        f'Indexed {total_indexed_files:,} '
        'numeric .npz files for '
        f'{len(frame_inventory):,} '
        'required sequences.'
    )
    print(
        f'Scanned {total_scanned_entries:,} '
        'total directory entries.'
    )
    print(
        f'Skipped {total_skipped_names:,} '
        'NPZ filenames without a clean '
        'numeric frame suffix.'
    )

    return frame_inventory


frame_inventory = build_frame_inventory(
    split_dfs
)


In [ ]:
def select_prior_frame_paths(
    target_path_value,
):
    target_path = resolve_frame_path(
        target_path_value
    )
    parsed_target = parse_frame_name(
        target_path
    )

    if parsed_target is None:
        raise ValueError(
            'Target filename does not end in a '
            'numeric frame index: '
            f'{target_path.name}'
        )

    sequence_id, target_frame_index = (
        parsed_target
    )
    inventory_key = (
        str(target_path.parent),
        sequence_id,
    )

    sequence_frames = frame_inventory.get(
        inventory_key
    )
    if sequence_frames is None:
        raise KeyError(
            'No frame inventory found for '
            f'{sequence_id} in '
            f'{target_path.parent}.'
        )

    # Strictly exclude the target and later frames.
    prior_frames = [
        (frame_index, frame_path)
        for frame_index, frame_path
        in sequence_frames
        if frame_index < target_frame_index
    ]

    recent_prior_frames = prior_frames[
        -MAX_LOOKBACK_FRAMES:
    ]
    available_in_window = len(
        recent_prior_frames
    )

    if available_in_window >= CLIP_LENGTH:
        sample_positions = np.linspace(
            0,
            available_in_window - 1,
            num=CLIP_LENGTH,
        )
        sample_positions = np.rint(
            sample_positions
        ).astype(int)
        selected_frames = [
            recent_prior_frames[position]
            for position in sample_positions
        ]
        padded_frame_count = 0
        used_black_padding = False

    elif available_in_window > 0:
        padded_frame_count = (
            CLIP_LENGTH
            - available_in_window
        )
        earliest_prior_frame = (
            recent_prior_frames[0]
        )
        selected_frames = (
            [earliest_prior_frame]
            * padded_frame_count
            + recent_prior_frames
        )
        used_black_padding = False

    else:
        padded_frame_count = CLIP_LENGTH
        selected_frames = [
            (None, None)
            for _ in range(CLIP_LENGTH)
        ]
        used_black_padding = True

    selected_indices = [
        frame_index
        for frame_index, _
        in selected_frames
        if frame_index is not None
    ]

    if any(
        frame_index >= target_frame_index
        for frame_index in selected_indices
    ):
        raise AssertionError(
            'A target or future frame was '
            f'selected for {target_path.name}.'
        )

    selected_paths = [
        frame_path
        for _, frame_path
        in selected_frames
    ]
    metadata = {
        'sequence_id': sequence_id,
        'target_frame_index': (
            target_frame_index
        ),
        'available_prior_frames': (
            len(prior_frames)
        ),
        'source_window_frames': (
            available_in_window
        ),
        'unique_selected_frames': len(
            set(selected_indices)
        ),
        'padded_frames': (
            padded_frame_count
        ),
        'used_black_padding': (
            used_black_padding
        ),
    }

    return selected_paths, metadata


clip_selections = {}
clip_metadata_dfs = {}

for split_name in SPLIT_NAMES:
    split_selections = []
    metadata_rows = []

    for row_index, row in (
        split_dfs[split_name].iterrows()
    ):
        selected_paths, metadata = (
            select_prior_frame_paths(
                row['target_frame_path']
            )
        )
        split_selections.append(
            selected_paths
        )
        metadata_rows.append({
            'split': split_name,
            'row_index': row_index,
            'target_frame_path': (
                row['target_frame_path']
            ),
            'class_label': (
                row['class_label']
            ),
            **metadata,
        })

    clip_selections[split_name] = (
        split_selections
    )
    clip_metadata_dfs[split_name] = (
        pd.DataFrame(metadata_rows)
    )

full_clip_metadata_df = pd.concat(
    clip_metadata_dfs.values(),
    ignore_index=True,
)

history_summary = (
    full_clip_metadata_df
    .groupby('class_label')[
        [
            'available_prior_frames',
            'unique_selected_frames',
            'padded_frames',
        ]
    ]
    .agg(['count', 'mean', 'median', 'min'])
    .reindex(CLASS_NAMES)
)
print(
    'Prior-frame availability by class:'
)
display(history_summary.round(2))

padding_rate = (
    full_clip_metadata_df
    .assign(
        padded=lambda frame: (
            frame['padded_frames'] > 0
        )
    )
    .groupby('class_label')['padded']
    .mean()
    .reindex(CLASS_NAMES)
    .mul(100)
    .rename('padded_percent')
)
display(padding_rate.to_frame().round(2))


# Step 6: Build and cache higher-resolution RGB clips

Each clip is cached as `uint8` to control disk usage. Batches are
normalized to `[0, 1]` only when read by the sequence loader, matching
the official MoViNet preprocessing convention.

The frame reader includes a camera-only compatibility path for `.npz`
files containing pickled StandardE2E objects, so the large StandardE2E
dependency stack does not need to be installed in this TensorFlow kernel.

The cache is published only after every row is complete. An interrupted
run therefore cannot be mistaken for a valid finished cache.


In [ ]:
# These .npz files contain pickled StandardE2E objects alongside the
# camera array. The transfer-learning environment intentionally omits the
# very large StandardE2E dependency stack. This restricted unpickler keeps
# NumPy's normal behavior for every other module and substitutes inert
# placeholders only for StandardE2E classes that are not used here.
_STANDARD_E2E_PLACEHOLDER_TYPES = {}
_USE_STANDARD_E2E_PICKLE_COMPAT = False


class _StandardE2EPlaceholder:
    def __new__(cls, *args, **kwargs):
        instance = super().__new__(cls)
        instance._constructor_args = args
        return instance

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
            return

        # Support the (instance_dict, slots_dict) state used by some
        # dataclasses and Pydantic-backed containers.
        if (
            isinstance(state, tuple)
            and len(state) == 2
        ):
            instance_state, slots_state = state
            if isinstance(instance_state, dict):
                self.__dict__.update(instance_state)
            if isinstance(slots_state, dict):
                self.__dict__.update(slots_state)
            return

        self._pickle_state = state


class _StandardE2ECompatUnpickler(pickle.Unpickler):
    def find_class(self, module_name, class_name):
        if (
            module_name == 'standard_e2e'
            or module_name.startswith('standard_e2e.')
        ):
            class_key = (module_name, class_name)
            if class_key not in _STANDARD_E2E_PLACEHOLDER_TYPES:
                _STANDARD_E2E_PLACEHOLDER_TYPES[class_key] = type(
                    class_name,
                    (_StandardE2EPlaceholder,),
                    {'__module__': module_name},
                )
            return _STANDARD_E2E_PLACEHOLDER_TYPES[class_key]

        return super().find_class(
            module_name,
            class_name,
        )


def _compat_pickle_load(file_object, **kwargs):
    return _StandardE2ECompatUnpickler(
        file_object,
        **kwargs,
    ).load()


@contextmanager
def _standard_e2e_pickle_compatibility():
    original_pickle_load = pickle.load
    pickle.load = _compat_pickle_load
    try:
        yield
    finally:
        pickle.load = original_pickle_load


def _read_modality_data(frame_path):
    global _USE_STANDARD_E2E_PICKLE_COMPAT

    def read_from_npz():
        with np.load(
            frame_path,
            allow_pickle=True,
        ) as frame_data:
            return frame_data[
                '_modality_data'
            ].item()

    if _USE_STANDARD_E2E_PICKLE_COMPAT:
        with _standard_e2e_pickle_compatibility():
            return read_from_npz()

    try:
        return read_from_npz()
    except ModuleNotFoundError as error:
        missing_module = error.name or ''
        if not missing_module.startswith('standard_e2e'):
            raise

        _USE_STANDARD_E2E_PICKLE_COMPAT = True
        print(
            'StandardE2E is not installed; using the '
            'camera-only compatibility loader.'
        )
        with _standard_e2e_pickle_compatibility():
            return read_from_npz()


def load_image(frame_path):
    # The first modality is the camera panorama.
    modality = _read_modality_data(
        frame_path
    )
    modality_keys = list(
        modality.keys()
    )
    image = np.asarray(
        modality[modality_keys[0]]
    )

    if np.issubdtype(
        image.dtype,
        np.floating,
    ):
        if image.max() <= 1.0:
            image = image * 255.0
        image = np.clip(
            image,
            0,
            255,
        ).astype(np.uint8)
    else:
        image = image.astype(np.uint8)

    return image


def prepare_frame(frame_path):
    image = load_image(frame_path)

    if image.ndim == 2:
        image = np.repeat(
            image[:, :, np.newaxis],
            3,
            axis=2,
        )

    if image.shape[-1] == 4:
        image = image[:, :, :3]

    if (
        image.ndim != 3
        or image.shape[-1] != 3
    ):
        raise ValueError(
            f'Unexpected image shape at '
            f'{frame_path}: {image.shape}'
        )

    image_tensor = tf.convert_to_tensor(
        image,
        dtype=tf.float32,
    )
    image_tensor = tf.image.resize(
        image_tensor,
        size=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
        ),
        method='bilinear',
        antialias=True,
    )

    return np.clip(
        np.rint(image_tensor.numpy()),
        0,
        255,
    ).astype(np.uint8)


def build_clip(selected_paths):
    if len(selected_paths) != CLIP_LENGTH:
        raise ValueError(
            f'Expected {CLIP_LENGTH} paths; '
            f'received {len(selected_paths)}.'
        )

    black_frame = np.zeros(
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
        dtype=np.uint8,
    )
    prepared_frame_cache = {}
    clip_frames = []

    for selected_path in selected_paths:
        if selected_path is None:
            clip_frames.append(black_frame)
            continue

        cache_key = str(selected_path)
        if cache_key not in (
            prepared_frame_cache
        ):
            prepared_frame_cache[
                cache_key
            ] = prepare_frame(
                selected_path
            )

        clip_frames.append(
            prepared_frame_cache[
                cache_key
            ]
        )

    return np.stack(clip_frames)


In [ ]:
current_cache_config = {
    'clip_length': CLIP_LENGTH,
    'max_lookback_frames': (
        MAX_LOOKBACK_FRAMES
    ),
    'image_height': IMAGE_HEIGHT,
    'image_width': IMAGE_WIDTH,
    'image_channels': IMAGE_CHANNELS,
    'resize_method': 'bilinear_panorama',
    'split_rows': {
        split_name: len(
            split_dfs[split_name]
        )
        for split_name in SPLIT_NAMES
    },
    'split_fingerprints': (
        split_fingerprints
    ),
}

estimated_cache_bytes = sum(
    len(split_dfs[split_name])
    * int(np.prod(INPUT_SHAPE))
    for split_name in SPLIT_NAMES
)
print(
    'Estimated total uint8 clip-cache '
    f'size: '
    f'{estimated_cache_bytes / (1024 ** 3):.3f} GB'
)

if CACHE_CONFIG_PATH.exists():
    with open(
        CACHE_CONFIG_PATH,
        'r',
        encoding='utf-8',
    ) as cache_config_file:
        saved_cache_config = json.load(
            cache_config_file
        )

    if (
        saved_cache_config
        != current_cache_config
        and not OVERWRITE_CLIP_CACHE
    ):
        raise ValueError(
            'Existing v2 clips use different '
            'settings. Set '
            'OVERWRITE_CLIP_CACHE=True to '
            'rebuild them.'
        )


def create_clip_cache(
    split_name,
    cache_path,
    selections,
    overwrite=False,
):
    expected_shape = (
        len(selections),
        *INPUT_SHAPE,
    )

    if (
        cache_path.exists()
        and not overwrite
    ):
        cached_array = np.load(
            cache_path,
            mmap_mode='r',
        )
        if (
            cached_array.shape
            != expected_shape
            or cached_array.dtype
            != np.uint8
        ):
            raise ValueError(
                f'{cache_path.name} has '
                f'{cached_array.shape} and '
                f'{cached_array.dtype}; '
                f'expected {expected_shape} '
                'and uint8.'
            )

        print(
            f'{cache_path.name} already '
            'exists with the expected shape.'
        )
        del cached_array
        return

    start_time = time.perf_counter()
    partial_cache_path = (
        cache_path.with_suffix(
            '.partial.npy'
        )
    )
    if partial_cache_path.exists():
        partial_cache_path.unlink()

    clip_cache = (
        np.lib.format.open_memmap(
            partial_cache_path,
            mode='w+',
            dtype=np.uint8,
            shape=expected_shape,
        )
    )

    for row_index, selected_paths in (
        enumerate(selections)
    ):
        clip_cache[row_index] = (
            build_clip(selected_paths)
        )

        completed_rows = row_index + 1
        if (
            completed_rows % 50 == 0
            or completed_rows
            == len(selections)
        ):
            print(
                f'  {split_name}: '
                f'{completed_rows:,}/'
                f'{len(selections):,} '
                'clips cached'
            )

    clip_cache.flush()
    del clip_cache
    partial_cache_path.replace(
        cache_path
    )

    elapsed_minutes = (
        time.perf_counter()
        - start_time
    ) / 60
    print(
        f'Created {cache_path.name} in '
        f'{elapsed_minutes:.2f} minutes'
    )


for split_name in SPLIT_NAMES:
    create_clip_cache(
        split_name=split_name,
        cache_path=(
            CLIP_CACHE_PATHS[
                split_name
            ]
        ),
        selections=(
            clip_selections[
                split_name
            ]
        ),
        overwrite=OVERWRITE_CLIP_CACHE,
    )
    clip_metadata_dfs[
        split_name
    ].to_csv(
        CLIP_METADATA_PATHS[
            split_name
        ],
        index=False,
    )

with open(
    CACHE_CONFIG_PATH,
    'w',
    encoding='utf-8',
) as cache_config_file:
    json.dump(
        current_cache_config,
        cache_config_file,
        indent=2,
    )

clip_arrays = {}
for split_name in SPLIT_NAMES:
    clip_array = np.load(
        CLIP_CACHE_PATHS[split_name],
        mmap_mode='r',
    )
    clip_arrays[split_name] = clip_array
    print(
        f'{split_name}: '
        f'shape={clip_array.shape}, '
        f'dtype={clip_array.dtype}, '
        f'disk='
        f'{clip_array.nbytes / (1024 ** 3):.3f} GB'
    )


# Step 7: Display one prior-only example GIF from each class

These GIFs verify chronological frame ordering and confirm that the
transfer model sees only historical context.


In [ ]:
GIF_FRAME_DURATION_SECONDS = 0.15

for class_id, class_name in enumerate(
    CLASS_NAMES
):
    matching_rows = np.flatnonzero(
        y_train == class_id
    )
    if len(matching_rows) == 0:
        raise ValueError(
            f'No training clips for '
            f'{class_name}.'
        )

    example_row = int(matching_rows[0])
    example_clip = np.asarray(
        clip_arrays['train'][
            example_row
        ],
        dtype=np.uint8,
    )
    gif_path = (
        GIF_OUTPUT_DIR
        / f'{class_name}_prior_clip.gif'
    )

    imageio.mimsave(
        gif_path,
        list(example_clip),
        duration=(
            GIF_FRAME_DURATION_SECONDS
        ),
        loop=0,
    )

    target_name = Path(
        split_dfs['train'].iloc[
            example_row
        ]['target_frame_path']
    ).name

    display(
        Markdown(
            f'### {class_name}\n'
            f'Target: `{target_name}`'
        )
    )
    display(
        IPythonImage(
            filename=str(gif_path)
        )
    )


# Step 8: Define a memory-safe clip loader

Clips remain memory-mapped as `uint8` and are converted to float RGB only
for the current batch. The loader supports clip-consistent brightness and
contrast augmentation, but frozen embedding extraction deliberately uses
the original clips so each sample needs only one backbone pass.


In [ ]:
class ClipSequence(keras.utils.Sequence):
    def __init__(
        self,
        cache_path,
        labels,
        batch_size,
        shuffle,
        seed,
        row_indices=None,
        augment=False,
    ):
        super().__init__()

        self.cache_path = Path(
            cache_path
        )
        self.clips = np.load(
            self.cache_path,
            mmap_mode='r',
        )
        self.labels = np.asarray(
            labels,
            dtype=np.int32,
        )
        self.batch_size = int(
            batch_size
        )
        self.shuffle = bool(shuffle)
        self.augment = bool(augment)
        self.rng = np.random.default_rng(
            seed
        )

        if row_indices is None:
            row_indices = np.arange(
                len(self.labels)
            )
        self.row_indices = np.asarray(
            row_indices,
            dtype=np.int64,
        )
        self.order = np.arange(
            len(self.row_indices)
        )

        if len(self.clips) != len(
            self.labels
        ):
            raise ValueError(
                f'{self.cache_path.name} '
                f'contains {len(self.clips)} '
                f'clips but received '
                f'{len(self.labels)} labels.'
            )

        if (
            self.row_indices.min() < 0
            or self.row_indices.max()
            >= len(self.labels)
        ):
            raise IndexError(
                'Sequence row indices are '
                'outside the cache bounds.'
            )

        self.on_epoch_end()

    def __len__(self):
        return math.ceil(
            len(self.row_indices)
            / self.batch_size
        )

    def __getitem__(self, batch_index):
        batch_start = (
            batch_index
            * self.batch_size
        )
        batch_end = min(
            batch_start
            + self.batch_size,
            len(self.row_indices),
        )
        local_positions = self.order[
            batch_start:batch_end
        ]
        cache_rows = self.row_indices[
            local_positions
        ]

        # MoViNet expects float RGB in [0, 1].
        batch_clips = np.asarray(
            self.clips[cache_rows],
            dtype=np.float32,
        ) / 255.0
        batch_labels = self.labels[
            cache_rows
        ]

        if self.augment:
            batch_clips = (
                self._augment_batch(
                    batch_clips
                )
            )

        return batch_clips, batch_labels

    def _augment_batch(
        self,
        batch_clips,
    ):
        batch_size = len(batch_clips)
        contrast = self.rng.uniform(
            1.0 - MAX_CONTRAST_DELTA,
            1.0 + MAX_CONTRAST_DELTA,
            size=(
                batch_size,
                1,
                1,
                1,
                1,
            ),
        ).astype(np.float32)
        brightness = self.rng.uniform(
            -MAX_BRIGHTNESS_DELTA,
            MAX_BRIGHTNESS_DELTA,
            size=(
                batch_size,
                1,
                1,
                1,
                1,
            ),
        ).astype(np.float32)

        channel_mean = batch_clips.mean(
            axis=(1, 2, 3),
            keepdims=True,
        )
        augmented = (
            (
                batch_clips
                - channel_mean
            )
            * contrast
            + channel_mean
            + brightness
        )

        return np.clip(
            augmented,
            0.0,
            1.0,
        ).astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.order)


def make_clip_sequence(
    split_name,
    batch_size,
    shuffle,
    seed,
    row_indices=None,
    augment=False,
):
    return ClipSequence(
        cache_path=(
            CLIP_CACHE_PATHS[
                split_name
            ]
        ),
        labels=y_by_split[split_name],
        batch_size=batch_size,
        shuffle=shuffle,
        seed=seed,
        row_indices=row_indices,
        augment=augment,
    )


inspection_sequence = (
    make_clip_sequence(
        split_name='train',
        batch_size=2,
        shuffle=False,
        seed=SEED,
        row_indices=(
            internal_training_indices[
                :2
            ]
        ),
        augment=True,
    )
)
inspection_clips, inspection_labels = (
    inspection_sequence[0]
)

print(
    f'Example batch shape: '
    f'{inspection_clips.shape}'
)
print(
    f'Example batch dtype: '
    f'{inspection_clips.dtype}'
)
print(
    'Example value range: '
    f'{inspection_clips.min():.3f} to '
    f'{inspection_clips.max():.3f}'
)
print(
    f'Example labels: '
    f'{inspection_labels}'
)

del inspection_sequence
del inspection_clips
gc.collect()


# Step 9: Define evaluation and plotting functions

Macro metrics give each class equal importance. Sensitivity is reported
as recall, including both macro and per-class recall. Model selection
prioritizes internal-validation macro F1, then left-turn recall, then
accuracy.


In [ ]:
def calculate_metrics(
    y_true,
    y_pred,
    model_name,
):
    recall_macro = recall_score(
        y_true,
        y_pred,
        average='macro',
        zero_division=0,
    )
    per_class_recall = recall_score(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        average=None,
        zero_division=0,
    )

    return {
        'model': model_name,
        'accuracy': accuracy_score(
            y_true,
            y_pred,
        ),
        'precision_macro': (
            precision_score(
                y_true,
                y_pred,
                average='macro',
                zero_division=0,
            )
        ),
        'recall_macro': recall_macro,
        'sensitivity_macro': (
            recall_macro
        ),
        'f1_macro': f1_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0,
        ),
        'straight_recall': float(
            per_class_recall[0]
        ),
        'right_turn_recall': float(
            per_class_recall[1]
        ),
        'left_turn_recall': float(
            per_class_recall[2]
        ),
        'precision_weighted': (
            precision_score(
                y_true,
                y_pred,
                average='weighted',
                zero_division=0,
            )
        ),
        'recall_weighted': recall_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0,
        ),
        'f1_weighted': f1_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0,
        ),
    }


def classification_report_df(
    y_true,
    y_pred,
):
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).T


def plot_confusion_matrices(
    y_true,
    y_pred,
    model_name,
):
    count_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
    )
    normalized_matrix = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=np.arange(
                NUM_CLASSES
            ),
            normalize='true',
        )
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5),
    )

    ConfusionMatrixDisplay(
        confusion_matrix=count_matrix,
        display_labels=CLASS_NAMES,
    ).plot(
        ax=axes[0],
        cmap='Blues',
        colorbar=False,
        values_format='d',
    )
    axes[0].set_title(
        f'{model_name}: Count Matrix'
    )
    axes[0].tick_params(
        axis='x',
        rotation=25,
    )

    ConfusionMatrixDisplay(
        confusion_matrix=(
            normalized_matrix
        ),
        display_labels=CLASS_NAMES,
    ).plot(
        ax=axes[1],
        cmap='Blues',
        colorbar=False,
        values_format='.2f',
    )
    axes[1].set_title(
        f'{model_name}: '
        'Row-Normalized Matrix'
    )
    axes[1].tick_params(
        axis='x',
        rotation=25,
    )

    plt.tight_layout()
    plt.show()


def plot_training_history(
    history,
    title,
):
    values = (
        history.history
        if hasattr(history, 'history')
        else history
    )
    epochs = np.arange(
        1,
        len(values['loss']) + 1,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 4),
    )
    axes[0].plot(
        epochs,
        values['loss'],
        marker='o',
        label='Training loss',
    )
    axes[0].plot(
        epochs,
        values['val_loss'],
        marker='o',
        label='Internal-validation loss',
    )
    axes[0].set_title(f'{title}: Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel(
        'Cross-Entropy Loss'
    )
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        epochs,
        values['accuracy'],
        marker='o',
        label='Training accuracy',
    )
    axes[1].plot(
        epochs,
        values['val_accuracy'],
        marker='o',
        label=(
            'Internal-validation accuracy'
        ),
    )
    axes[1].set_title(
        f'{title}: Accuracy'
    )
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def count_model_parameters(model):
    trainable_parameters = int(sum(
        np.prod(variable.shape)
        for variable
        in model.trainable_weights
    ))
    non_trainable_parameters = int(sum(
        np.prod(variable.shape)
        for variable
        in model.non_trainable_weights
    ))

    return {
        'total_parameters': int(
            model.count_params()
        ),
        'trainable_parameters': (
            trainable_parameters
        ),
        'non_trainable_parameters': (
            non_trainable_parameters
        ),
    }


# Step 10: Download and restore the Kinetics-600 MoViNet-A0 checkpoint

This is the only network download. The checkpoint is reused from the local
`pretrained` directory on later runs.


In [ ]:
def locate_movinet_checkpoint():
    archive_path = Path(
        keras.utils.get_file(
            fname=(
                MOVINET_CHECKPOINT_FILENAME
            ),
            origin=(
                MOVINET_CHECKPOINT_URL
            ),
            cache_dir=str(
                CNN_3D_V2_DATA_DIR
            ),
            cache_subdir='pretrained',
            extract=False,
        )
    )

    checkpoint_index_files = [
        path
        for path in (
            archive_path.parent.rglob(
                'ckpt-*.index'
            )
        )
        if (
            f'movinet_{MOVINET_MODEL_ID}_'
            f'{MOVINET_VARIANT}'
            in str(path.parent)
        )
    ]

    if not checkpoint_index_files:
        # The official URL has a .tar.gz suffix but currently
        # serves an uncompressed POSIX tar archive. "r:*"
        # detects either representation correctly.
        with tarfile.open(
            archive_path,
            mode='r:*',
        ) as archive:
            archive.extractall(
                path=archive_path.parent
            )

        checkpoint_index_files = [
            path
            for path in (
                archive_path.parent.rglob(
                    'ckpt-*.index'
                )
            )
            if (
                f'movinet_{MOVINET_MODEL_ID}_'
                f'{MOVINET_VARIANT}'
                in str(path.parent)
            )
        ]

    if not checkpoint_index_files:
        raise FileNotFoundError(
            'The MoViNet archive was '
            'downloaded, but no checkpoint '
            'index was found beneath '
            f'{archive_path.parent}.'
        )

    checkpoint_index_path = sorted(
        checkpoint_index_files
    )[-1]

    # TensorFlow expects the prefix without ".index".
    checkpoint_path = str(
        checkpoint_index_path
    )[:-len('.index')]

    print(
        'MoViNet checkpoint: '
        f'{checkpoint_path}'
    )
    return checkpoint_path


MOVINET_CHECKPOINT_PATH = (
    locate_movinet_checkpoint()
)


def load_kinetics_backbone():
    # The official streaming A0 architecture uses causal (2+1)D
    # operations backed by optimized Conv2D kernels. It can still process
    # the complete prior-only clip in one call and is substantially more
    # CPU-friendly than the base model's Conv3D implementation.
    backbone = movinet.Movinet(
        model_id=MOVINET_MODEL_ID,
        causal=True,
        conv_type='2plus1d',
        se_type='2plus3d',
        activation='hard_swish',
        gating_activation='hard_sigmoid',
        use_positional_encoding=False,
        use_external_states=False,
        # MovinetClassifier always expects the backbone to return the
        # pair (endpoints, states), even though the classifier itself
        # returns logits only. Keep backbone state output enabled.
        output_states=True,
    )

    kinetics_classifier = (
        movinet_model.MovinetClassifier(
            backbone=backbone,
            num_classes=600,
            output_states=False,
        )
    )
    kinetics_classifier.build([
        None,
        CLIP_LENGTH,
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        IMAGE_CHANNELS,
    ])

    checkpoint = tf.train.Checkpoint(
        model=kinetics_classifier
    )
    restore_status = checkpoint.restore(
        MOVINET_CHECKPOINT_PATH
    )
    restore_status.assert_existing_objects_matched()

    backbone.trainable = False
    print(
        'Restored Kinetics-600 weights '
        f'into MoViNet-{MOVINET_MODEL_ID.upper()}-Stream.'
    )

    return backbone


# Step 11: Build the frozen MoViNet video-embedding extractor

The Kinetics classifier is used only to restore the official weights.
We then expose MoViNet's final video representation and globally pool
its spatiotemporal grid into one vector per prior-frame clip. The
backbone remains frozen throughout this CPU workflow.


In [ ]:
EMBEDDING_BATCH_SIZE = 2


def build_movinet_embedding_extractor():
    backbone = load_kinetics_backbone()

    video_input = keras.Input(
        shape=INPUT_SHAPE,
        name='prior_rgb_clip',
    )
    endpoints, _ = backbone({
        'image': video_input,
    })

    if 'head' not in endpoints:
        raise KeyError(
            'MoViNet did not return the expected "head" endpoint. '
            f'Available endpoints: {list(endpoints)}'
        )

    video_embedding = (
        keras.layers.GlobalAveragePooling3D(
            name='movinet_global_average_pool',
        )(endpoints['head'])
    )

    extractor = keras.Model(
        inputs=video_input,
        outputs=video_embedding,
        name='movinet_a0_embedding_extractor',
    )
    extractor.trainable = False

    return extractor, backbone


keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

embedding_extractor, embedding_backbone = (
    build_movinet_embedding_extractor()
)
EMBEDDING_DIM = int(
    embedding_extractor.output_shape[-1]
)

embedding_extractor.summary(
    expand_nested=False
)
print(
    f'Frozen MoViNet embedding dimension: '
    f'{EMBEDDING_DIM:,}'
)


# Step 12: Extract and cache one MoViNet embedding per clip

This is the expensive step on the first execution. Each split is saved
independently, validated, and reused on later executions. If the run is
interrupted after a split completes, rerunning the cell skips that
completed split.

Embeddings are extracted without shuffle or augmentation, so row `i` in
each embedding file corresponds exactly to row `i` in that split's
`df_*.csv` and `y_*.csv`.


In [ ]:
@tf.function(
    input_signature=[
        tf.TensorSpec(
            shape=(None, *INPUT_SHAPE),
            dtype=tf.float32,
        )
    ]
)
def extract_embedding_batch(batch_clips):
    # Calling the model directly preserves every non-batch dimension.
    # The legacy Keras Sequence adapter otherwise reports a completely
    # unknown 5-D shape to MoViNet's stem.
    return embedding_extractor(
        batch_clips,
        training=False,
    )


current_embedding_cache_config = {
    'model_id': MOVINET_MODEL_ID,
    'model_variant': MOVINET_VARIANT,
    'conv_type': '2plus1d',
    'pretraining': 'Kinetics-600',
    'pooling': 'global_average_pool_movinet_head',
    'clip_length': CLIP_LENGTH,
    'max_lookback_frames': MAX_LOOKBACK_FRAMES,
    'image_height': IMAGE_HEIGHT,
    'image_width': IMAGE_WIDTH,
    'image_channels': IMAGE_CHANNELS,
    'embedding_dim': EMBEDDING_DIM,
    'split_rows': {
        split_name: len(split_dfs[split_name])
        for split_name in SPLIT_NAMES
    },
    'split_fingerprints': split_fingerprints,
}

if EMBEDDING_CONFIG_PATH.exists():
    with open(
        EMBEDDING_CONFIG_PATH,
        'r',
        encoding='utf-8',
    ) as embedding_config_file:
        saved_embedding_cache_config = (
            json.load(embedding_config_file)
        )

    if (
        saved_embedding_cache_config
        != current_embedding_cache_config
        and not OVERWRITE_EMBEDDING_CACHE
    ):
        raise ValueError(
            'Existing MoViNet embeddings use different clips, '
            'splits, or model settings. Set '
            'OVERWRITE_EMBEDDING_CACHE=True to rebuild them.'
        )


def validate_embedding_array(
    embeddings,
    split_name,
):
    expected_rows = len(
        split_dfs[split_name]
    )

    if embeddings.shape != (
        expected_rows,
        EMBEDDING_DIM,
    ):
        raise ValueError(
            f'{split_name} embeddings have shape '
            f'{embeddings.shape}; expected '
            f'{(expected_rows, EMBEDDING_DIM)}.'
        )

    if embeddings.dtype != np.float32:
        raise TypeError(
            f'{split_name} embeddings have dtype '
            f'{embeddings.dtype}; expected float32.'
        )

    if not np.isfinite(embeddings).all():
        raise ValueError(
            f'{split_name} embeddings contain NaN or '
            'infinite values.'
        )


def create_embedding_cache(
    split_name,
    overwrite=False,
):
    cache_path = EMBEDDING_CACHE_PATHS[
        split_name
    ]

    if (
        cache_path.exists()
        and not overwrite
    ):
        cached_embeddings = np.load(
            cache_path,
        )
        validate_embedding_array(
            cached_embeddings,
            split_name,
        )
        print(
            f'{cache_path.name} already exists '
            f'with shape {cached_embeddings.shape}.'
        )
        del cached_embeddings
        return

    print(
        f'\nExtracting frozen MoViNet embeddings '
        f'for {split_name}...'
    )
    start_time = time.perf_counter()

    clip_sequence = make_clip_sequence(
        split_name=split_name,
        batch_size=EMBEDDING_BATCH_SIZE,
        shuffle=False,
        seed=SEED,
        augment=False,
    )
    expected_rows = len(
        split_dfs[split_name]
    )
    total_batches = len(clip_sequence)
    partial_cache_path = (
        cache_path.with_suffix(
            '.partial.npy'
        )
    )
    if partial_cache_path.exists():
        partial_cache_path.unlink()

    embedding_cache = (
        np.lib.format.open_memmap(
            partial_cache_path,
            mode='w+',
            dtype=np.float32,
            shape=(
                expected_rows,
                EMBEDDING_DIM,
            ),
        )
    )
    row_start = 0

    try:
        for batch_index in range(
            total_batches
        ):
            batch_clips, _ = (
                clip_sequence[batch_index]
            )
            batch_tensor = tf.ensure_shape(
                tf.convert_to_tensor(
                    batch_clips,
                    dtype=tf.float32,
                ),
                (None, *INPUT_SHAPE),
            )
            batch_embeddings = np.asarray(
                extract_embedding_batch(
                    batch_tensor
                ).numpy(),
                dtype=np.float32,
            )

            expected_batch_shape = (
                len(batch_clips),
                EMBEDDING_DIM,
            )
            if (
                batch_embeddings.shape
                != expected_batch_shape
            ):
                raise ValueError(
                    'MoViNet returned embeddings '
                    f'with shape '
                    f'{batch_embeddings.shape}; '
                    f'expected '
                    f'{expected_batch_shape}.'
                )

            row_end = (
                row_start
                + len(batch_embeddings)
            )
            embedding_cache[
                row_start:row_end
            ] = batch_embeddings
            row_start = row_end

            completed_batches = (
                batch_index + 1
            )
            if (
                completed_batches % 10 == 0
                or completed_batches
                == total_batches
            ):
                elapsed_seconds = max(
                    time.perf_counter()
                    - start_time,
                    1e-9,
                )
                clips_per_second = (
                    row_end
                    / elapsed_seconds
                )
                print(
                    f'  {split_name}: '
                    f'{completed_batches:,}/'
                    f'{total_batches:,} batches; '
                    f'{row_end:,}/'
                    f'{expected_rows:,} clips; '
                    f'{clips_per_second:.2f} clips/s'
                )

        if row_start != expected_rows:
            raise ValueError(
                f'{split_name} extraction wrote '
                f'{row_start:,} rows; expected '
                f'{expected_rows:,}.'
            )

        embedding_cache.flush()
    finally:
        del embedding_cache

    partial_embeddings = np.load(
        partial_cache_path,
        mmap_mode='r',
    )
    validate_embedding_array(
        partial_embeddings,
        split_name,
    )
    del partial_embeddings

    partial_cache_path.replace(
        cache_path
    )

    elapsed_minutes = (
        time.perf_counter()
        - start_time
    ) / 60
    print(
        f'Saved {cache_path.name} in '
        f'{elapsed_minutes:.2f} minutes'
    )

    del clip_sequence
    gc.collect()


for split_name in SPLIT_NAMES:
    create_embedding_cache(
        split_name=split_name,
        overwrite=(
            OVERWRITE_EMBEDDING_CACHE
        ),
    )

with open(
    EMBEDDING_CONFIG_PATH,
    'w',
    encoding='utf-8',
) as embedding_config_file:
    json.dump(
        current_embedding_cache_config,
        embedding_config_file,
        indent=2,
    )

embeddings_by_split = {}
for split_name in SPLIT_NAMES:
    split_embeddings = np.load(
        EMBEDDING_CACHE_PATHS[
            split_name
        ],
    )
    validate_embedding_array(
        split_embeddings,
        split_name,
    )
    embeddings_by_split[
        split_name
    ] = split_embeddings
    print(
        f'{split_name}: '
        f'{split_embeddings.shape}, '
        f'{split_embeddings.nbytes / (1024 ** 2):.2f} MB'
    )

# The expensive backbone is no longer needed after the caches exist.
del embedding_extractor
del embedding_backbone
keras.backend.clear_session()
gc.collect()

print(
    '\nFrozen MoViNet extraction is complete. '
    'All remaining training uses cached vectors.'
)


# Step 13: Put the custom classification head into a tunable function

The head uses layer normalization, one learned hidden representation,
dropout, and a three-class softmax output. The `dense_hidden` layer is
the final feature vector exported at the end of the notebook.

Tunable hyperparameters:

- hidden-layer width;
- dropout;
- learning rate;
- L2 weight decay;
- batch size;
- optional class weighting.

Model selection prioritizes internal-validation macro F1, with
internal-validation loss as the tie-breaker.


In [ ]:
MODEL_CONFIG_KEYS = [
    'dense_units',
    'dropout_rate',
    'learning_rate',
    'l2_strength',
    'batch_size',
    'use_class_weights',
]

LEFT_TURN_CLASS_ID = CLASS_TO_ID[
    'left-turn'
]
LEFT_TURN_MULTIPLIER_GRID = np.round(
    np.linspace(
        1.0,
        2.5,
        31,
    ),
    2,
)


def apply_left_turn_probability_multiplier(
    probabilities,
    multiplier,
):
    adjusted_probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    ).copy()

    if adjusted_probabilities.ndim != 2:
        raise ValueError(
            'Probabilities must be a two-dimensional array.'
        )
    if adjusted_probabilities.shape[1] != NUM_CLASSES:
        raise ValueError(
            'Probability columns do not match NUM_CLASSES.'
        )
    if not np.isfinite(adjusted_probabilities).all():
        raise ValueError(
            'Probabilities contain NaN or infinite values.'
        )
    if multiplier <= 0:
        raise ValueError(
            'The left-turn multiplier must be positive.'
        )

    adjusted_probabilities[
        :,
        LEFT_TURN_CLASS_ID,
    ] *= float(multiplier)

    probability_totals = (
        adjusted_probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )
    if np.any(probability_totals <= 0):
        raise ValueError(
            'Adjusted probability rows must have positive sums.'
        )

    return (
        adjusted_probabilities
        / probability_totals
    ).astype(np.float32)


def tune_left_turn_probability_multiplier(
    y_true,
    probabilities,
):
    tuning_rows = []

    for multiplier in (
        LEFT_TURN_MULTIPLIER_GRID
    ):
        adjusted_probabilities = (
            apply_left_turn_probability_multiplier(
                probabilities,
                multiplier,
            )
        )
        adjusted_predictions = np.argmax(
            adjusted_probabilities,
            axis=1,
        )
        metrics = calculate_metrics(
            y_true,
            adjusted_predictions,
            'Internal multiplier candidate',
        )
        tuning_rows.append({
            'left_turn_multiplier': float(
                multiplier
            ),
            'accuracy': metrics['accuracy'],
            'precision_macro': metrics[
                'precision_macro'
            ],
            'recall_macro': metrics[
                'recall_macro'
            ],
            'f1_macro': metrics['f1_macro'],
            'left_turn_recall': metrics[
                'left_turn_recall'
            ],
        })

    tuning_df = pd.DataFrame(
        tuning_rows
    )
    ranked_tuning_df = (
        tuning_df.sort_values(
            by=[
                'f1_macro',
                'left_turn_multiplier',
            ],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    # When several multipliers yield the same predictions and score,
    # select the smallest adjustment.
    selected_multiplier = float(
        ranked_tuning_df.loc[
            0,
            'left_turn_multiplier',
        ]
    )
    return (
        selected_multiplier,
        tuning_df,
    )


def plot_left_turn_multiplier_search(
    tuning_df,
    selected_multiplier,
    title,
):
    fig, ax = plt.subplots(
        figsize=(9, 5),
    )
    ax.plot(
        tuning_df['left_turn_multiplier'],
        tuning_df['f1_macro'],
        marker='o',
        label='Macro F1',
    )
    ax.plot(
        tuning_df['left_turn_multiplier'],
        tuning_df['left_turn_recall'],
        marker='o',
        label='Left-turn recall',
    )
    ax.plot(
        tuning_df['left_turn_multiplier'],
        tuning_df['accuracy'],
        marker='o',
        label='Accuracy',
    )
    ax.axvline(
        selected_multiplier,
        color='black',
        linestyle='--',
        label=(
            'Selected multiplier = '
            f'{selected_multiplier:.2f}'
        ),
    )
    ax.set_title(
        f'{title}: Left-Turn Decision Tuning'
    )
    ax.set_xlabel(
        'Left-turn probability multiplier'
    )
    ax.set_ylabel('Validation score')
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


def compile_head_model(
    model,
    learning_rate,
):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=learning_rate,
        ),
        loss=(
            'sparse_categorical_crossentropy'
        ),
        metrics=['accuracy'],
    )
    return model


def build_head_model(config):
    embedding_input = keras.Input(
        shape=(EMBEDDING_DIM,),
        name='movinet_embedding',
    )
    x = keras.layers.LayerNormalization(
        name='embedding_layer_norm',
    )(embedding_input)
    x = keras.layers.Dense(
        units=config['dense_units'],
        activation='relu',
        kernel_regularizer=(
            keras.regularizers.l2(
                config['l2_strength']
            )
        ),
        name='dense_hidden',
    )(x)
    x = keras.layers.Dropout(
        rate=config['dropout_rate'],
        name='dense_dropout',
    )(x)
    output = keras.layers.Dense(
        units=NUM_CLASSES,
        activation='softmax',
        name='class_probabilities',
    )(x)

    model = keras.Model(
        inputs=embedding_input,
        outputs=output,
        name='waymo_movinet_cached_head',
    )
    return compile_head_model(
        model,
        config['learning_rate'],
    )


def selection_score(result_record):
    return (
        result_record[
            'internal_val_f1_macro'
        ],
        -result_record[
            'internal_val_loss'
        ],
    )


def evaluate_internal_validation(
    model,
    config_name,
    batch_size,
):
    validation_probabilities = (
        model.predict(
            embeddings_by_split['train'][
                internal_validation_indices
            ],
            batch_size=batch_size,
            verbose=0,
        )
    )
    unadjusted_predictions = np.argmax(
        validation_probabilities,
        axis=1,
    )
    unadjusted_metrics = calculate_metrics(
        y_internal_validation,
        unadjusted_predictions,
        (
            'Internal validation unadjusted: '
            f'{config_name}'
        ),
    )

    (
        selected_multiplier,
        multiplier_tuning_df,
    ) = tune_left_turn_probability_multiplier(
        y_internal_validation,
        validation_probabilities,
    )
    adjusted_probabilities = (
        apply_left_turn_probability_multiplier(
            validation_probabilities,
            selected_multiplier,
        )
    )
    validation_predictions = np.argmax(
        adjusted_probabilities,
        axis=1,
    )
    validation_metrics = calculate_metrics(
        y_internal_validation,
        validation_predictions,
        (
            'Internal validation adjusted: '
            f'{config_name}'
        ),
    )
    return (
        validation_metrics,
        validation_predictions,
        selected_multiplier,
        multiplier_tuning_df,
        unadjusted_metrics,
    )


def train_and_score_head(
    config_name,
    config,
    epochs,
    patience,
    verbose=2,
):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)

    model = build_head_model(
        config
    )
    class_weight = (
        internal_class_weights
        if config['use_class_weights']
        else None
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=max(2, patience // 2),
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    start_time = time.perf_counter()
    history = model.fit(
        embeddings_by_split['train'][
            internal_training_indices
        ],
        y_internal_training,
        validation_data=(
            embeddings_by_split['train'][
                internal_validation_indices
            ],
            y_internal_validation,
        ),
        epochs=epochs,
        batch_size=config['batch_size'],
        shuffle=True,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=verbose,
    )
    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    (
        validation_metrics,
        validation_predictions,
        selected_multiplier,
        multiplier_tuning_df,
        unadjusted_metrics,
    ) = evaluate_internal_validation(
        model=model,
        config_name=config_name,
        batch_size=config[
            'batch_size'
        ],
    )

    best_epoch_index = int(
        np.argmin(
            history.history['val_loss']
        )
    )
    parameter_counts = (
        count_model_parameters(
            model
        )
    )

    result_record = {
        'config_name': config_name,
        'best_epoch': (
            best_epoch_index + 1
        ),
        'epochs_ran': len(
            history.history['loss']
        ),
        'training_seconds': (
            elapsed_seconds
        ),
        'internal_val_loss': float(
            history.history['val_loss'][
                best_epoch_index
            ]
        ),
        'left_turn_multiplier': (
            selected_multiplier
        ),
        'internal_val_accuracy_unadjusted': (
            unadjusted_metrics[
                'accuracy'
            ]
        ),
        'internal_val_f1_macro_unadjusted': (
            unadjusted_metrics[
                'f1_macro'
            ]
        ),
        'internal_val_left_turn_recall_unadjusted': (
            unadjusted_metrics[
                'left_turn_recall'
            ]
        ),
        'internal_val_accuracy': (
            validation_metrics[
                'accuracy'
            ]
        ),
        'internal_val_precision_macro': (
            validation_metrics[
                'precision_macro'
            ]
        ),
        'internal_val_recall_macro': (
            validation_metrics[
                'recall_macro'
            ]
        ),
        'internal_val_f1_macro': (
            validation_metrics[
                'f1_macro'
            ]
        ),
        'internal_val_left_turn_recall': (
            validation_metrics[
                'left_turn_recall'
            ]
        ),
        **parameter_counts,
        **{
            key: config[key]
            for key in MODEL_CONFIG_KEYS
        },
    }

    return (
        model,
        history,
        validation_predictions,
        result_record,
        multiplier_tuning_df,
    )


# Step 14: Run one initial head and multiplier trial

This trial verifies that the cached embeddings, labels, optimization
path, metrics, multiplier search, and confusion-matrix code all work
before the tuning loop. Because the backbone is no longer executed during
epochs, verbose per-epoch output is inexpensive.


In [ ]:
HEAD_TUNING_EPOCHS = 80
HEAD_EARLY_STOPPING_PATIENCE = 10

INITIAL_HEAD_CONFIG = {
    'dense_units': 128,
    'dropout_rate': 0.50,
    'learning_rate': 0.001,
    'l2_strength': 0.0001,
    'batch_size': 32,
    'use_class_weights': True,
}

(
    initial_head_model,
    initial_head_history,
    initial_head_predictions,
    initial_head_result,
    initial_multiplier_results,
) = train_and_score_head(
    config_name='Initial 128-unit head',
    config=INITIAL_HEAD_CONFIG,
    epochs=HEAD_TUNING_EPOCHS,
    patience=(
        HEAD_EARLY_STOPPING_PATIENCE
    ),
    verbose=2,
)

initial_head_model.summary()
plot_training_history(
    initial_head_history,
    'Initial Frozen-MoViNet Head',
)
display(
    pd.DataFrame([
        initial_head_result
    ]).set_index('config_name')
)
print(
    'Selected initial left-turn multiplier: '
    f'{initial_head_result["left_turn_multiplier"]:.2f}'
)
plot_left_turn_multiplier_search(
    initial_multiplier_results,
    initial_head_result[
        'left_turn_multiplier'
    ],
    'Initial Head',
)
display(
    classification_report_df(
        y_internal_validation,
        initial_head_predictions,
    )
)
plot_confusion_matrices(
    y_internal_validation,
    initial_head_predictions,
    'Initial Head Adjusted — Internal Validation',
)

best_config = dict(
    INITIAL_HEAD_CONFIG
)
best_head_result = dict(
    initial_head_result
)
best_head_history = (
    initial_head_history
)
best_multiplier_results = (
    initial_multiplier_results.copy()
)

tuning_results = [
    initial_head_result
]

del initial_head_model
gc.collect()


# Step 15: Tune the dense head and left-turn multiplier

The grid is deliberately compact and targets the hyperparameters most
likely to matter with only about two thousand examples. It compares
hidden capacity, regularization, learning rate, and class weighting. For
each trained head, the same fixed multiplier grid is evaluated on the
train-internal validation probabilities. No shared-validation or test
rows are used here.


In [ ]:
HEAD_TUNING_CONFIGS = [
    {
        'config_name': (
            'Smaller regularized head'
        ),
        'dense_units': 64,
        'dropout_rate': 0.40,
        'learning_rate': 0.001,
        'l2_strength': 0.0001,
        'batch_size': 32,
        'use_class_weights': True,
    },
    {
        'config_name': (
            'Lower learning rate'
        ),
        'dense_units': 128,
        'dropout_rate': 0.50,
        'learning_rate': 0.0003,
        'l2_strength': 0.0001,
        'batch_size': 32,
        'use_class_weights': True,
    },
    {
        'config_name': (
            'Larger strongly regularized head'
        ),
        'dense_units': 256,
        'dropout_rate': 0.60,
        'learning_rate': 0.0003,
        'l2_strength': 0.0005,
        'batch_size': 32,
        'use_class_weights': True,
    },
    {
        'config_name': (
            'Unweighted control'
        ),
        'dense_units': 128,
        'dropout_rate': 0.50,
        'learning_rate': 0.0003,
        'l2_strength': 0.0001,
        'batch_size': 32,
        'use_class_weights': False,
    },
]

display(
    pd.DataFrame(
        HEAD_TUNING_CONFIGS
    ).set_index('config_name')
)


In [ ]:
for trial_number, trial_spec in enumerate(
    HEAD_TUNING_CONFIGS,
    start=1,
):
    config_name = trial_spec[
        'config_name'
    ]
    trial_config = {
        key: trial_spec[key]
        for key in MODEL_CONFIG_KEYS
    }

    print(
        '\n'
        + '=' * 80
    )
    print(
        f'Head tuning trial '
        f'{trial_number}/'
        f'{len(HEAD_TUNING_CONFIGS)}: '
        f'{config_name}'
    )
    print(trial_config)

    (
        trial_head_model,
        trial_head_history,
        trial_predictions,
        trial_result,
        trial_multiplier_results,
    ) = train_and_score_head(
        config_name=config_name,
        config=trial_config,
        epochs=HEAD_TUNING_EPOCHS,
        patience=(
            HEAD_EARLY_STOPPING_PATIENCE
        ),
        verbose=2,
    )

    tuning_results.append(
        trial_result
    )
    display(
        pd.DataFrame([
            trial_result
        ]).set_index('config_name')
    )
    plot_training_history(
        trial_head_history,
        config_name,
    )
    plot_left_turn_multiplier_search(
        trial_multiplier_results,
        trial_result[
            'left_turn_multiplier'
        ],
        config_name,
    )

    if (
        selection_score(trial_result)
        > selection_score(
            best_head_result
        )
    ):
        best_config = dict(
            trial_config
        )
        best_head_result = dict(
            trial_result
        )
        best_head_history = (
            trial_head_history
        )
        best_multiplier_results = (
            trial_multiplier_results.copy()
        )
        print(
            'New best head selected by '
            'adjusted internal-validation macro F1.'
        )

    del trial_head_model
    del trial_predictions
    del trial_multiplier_results
    gc.collect()


In [ ]:
tuning_results_df = (
    pd.DataFrame(
        tuning_results
    )
    .sort_values(
        by=[
            'internal_val_f1_macro',
            'internal_val_loss',
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    'Head and multiplier tuning results, ordered by '
    'selection criterion:'
)
display(
    tuning_results_df.set_index(
        'config_name'
    )
)

print(
    '\nSelected head configuration:'
)
print(best_config)
print(
    'Selected left-turn multiplier: '
    f'{best_head_result["left_turn_multiplier"]:.2f}'
)
print(
    'Selected internal-validation '
    f'macro F1: '
    f'{best_head_result["internal_val_f1_macro"]:.4f}'
)
print(
    'Selected best epoch: '
    f'{best_head_result["best_epoch"]}'
)

print(
    '\nTop multiplier candidates for the selected head:'
)
display(
    best_multiplier_results.sort_values(
        by=[
            'f1_macro',
            'left_turn_multiplier',
        ],
        ascending=[False, True],
    ).head(10)
)


# Step 16: Lock the CPU execution plan before final training

End-to-end backbone fine-tuning is intentionally disabled in this
version. On this laptop it would repeat MoViNet's full spatiotemporal
computation for every batch and epoch, defeating the purpose of the
cache. More importantly, it is not required to obtain a clean,
Kinetics-pretrained video feature benchmark.

The selected head hyperparameters, epoch count, and left-turn multiplier
are now frozen. The shared validation and test folds still have not
influenced selection.


In [ ]:
BACKBONE_FINE_TUNING_PERFORMED = False
selected_training_plan = (
    'Frozen Kinetics-600 MoViNet-A0 embeddings '
    '+ tuned custom dense head'
)
selected_head_epochs = max(
    1,
    int(
        best_head_result[
            'best_epoch'
        ]
    ),
)
selected_left_turn_multiplier = float(
    best_head_result[
        'left_turn_multiplier'
    ]
)

print(
    f'Selected plan: '
    f'{selected_training_plan}'
)
print(
    'Backbone fine-tuning performed: '
    f'{BACKBONE_FINE_TUNING_PERFORMED}'
)
print(
    'Final head epochs learned only '
    f'from the training fold: '
    f'{selected_head_epochs}'
)
print(
    'Locked left-turn probability multiplier: '
    f'{selected_left_turn_multiplier:.2f}'
)
print(
    'Multiplier selected using only '
    'train-internal validation: True'
)
print(
    'Shared validation used for '
    'selection: False'
)
print(
    'Test used for selection: False'
)


# Step 17: Retrain the selected head on the full official training fold

Early stopping is no longer needed because the epoch count was learned
from the train-internal validation experiment. The final head is
initialized from scratch and trained on every official training row.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)

final_model = build_head_model(
    best_config
)
final_model.summary()

final_class_weight = (
    full_training_class_weights
    if best_config[
        'use_class_weights'
    ]
    else None
)

final_training_start = (
    time.perf_counter()
)
final_head_history = final_model.fit(
    embeddings_by_split['train'],
    y_train,
    epochs=selected_head_epochs,
    batch_size=best_config[
        'batch_size'
    ],
    shuffle=True,
    class_weight=final_class_weight,
    verbose=2,
)
final_training_seconds = (
    time.perf_counter()
    - final_training_start
)

final_parameter_counts = (
    count_model_parameters(
        final_model
    )
)
print(
    '\nFinal head parameter counts:'
)
print(final_parameter_counts)
print(
    'Final head training time: '
    f'{final_training_seconds:.1f} seconds'
)

final_history_df = pd.DataFrame(
    final_head_history.history
)
display(
    final_history_df.tail(10)
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4),
)
axes[0].plot(
    final_history_df['loss'],
    label='Training loss',
)
axes[0].set_title(
    'Final Full-Training Loss'
)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.25)

axes[1].plot(
    final_history_df['accuracy'],
    label='Training accuracy',
)
axes[1].set_title(
    'Final Full-Training Accuracy'
)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


# Step 18: Report the untouched shared-validation fold

This fold is reported for comparison with the earlier CNN notebooks.
It was not used for head tuning, early stopping, or epoch selection.


In [ ]:
shared_validation_evaluation = (
    final_model.evaluate(
        embeddings_by_split['val'],
        y_val,
        batch_size=best_config[
            'batch_size'
        ],
        verbose=1,
        return_dict=True,
    )
)
shared_validation_probabilities = (
    final_model.predict(
        embeddings_by_split['val'],
        batch_size=best_config[
            'batch_size'
        ],
        verbose=1,
    )
)
shared_validation_unadjusted_predictions = (
    np.argmax(
        shared_validation_probabilities,
        axis=1,
    )
)

shared_validation_adjusted_probabilities = (
    apply_left_turn_probability_multiplier(
        shared_validation_probabilities,
        selected_left_turn_multiplier,
    )
)
shared_validation_predictions = np.argmax(
    shared_validation_adjusted_probabilities,
    axis=1,
)

shared_validation_unadjusted_metrics = (
    calculate_metrics(
        y_val,
        shared_validation_unadjusted_predictions,
        'MoViNet Head Validation — Unadjusted',
    )
)
shared_validation_metrics = (
    calculate_metrics(
        y_val,
        shared_validation_predictions,
        (
            'MoViNet Head Validation — '
            'Adjusted'
        ),
    )
)

print(
    'Keras validation evaluation:'
)
print(shared_validation_evaluation)
print(
    'Locked left-turn multiplier applied: '
    f'{selected_left_turn_multiplier:.2f}'
)
display(
    pd.DataFrame([
        shared_validation_unadjusted_metrics,
        shared_validation_metrics,
    ]).set_index('model')
)
display(
    classification_report_df(
        y_val,
        shared_validation_predictions,
    )
)
plot_confusion_matrices(
    y_val,
    shared_validation_predictions,
    'Selected MoViNet Head Validation — Adjusted',
)


# Step 19: Define chance and majority-class baselines

Both baselines are evaluated with the same metric function and class
order as the transfer-learning model.


In [ ]:
majority_class_id = int(
    np.bincount(y_train).argmax()
)
class_probabilities = (
    np.bincount(
        y_train,
        minlength=NUM_CLASSES,
    )
    / len(y_train)
)


def baseline_predictions(
    y_reference,
    seed=SEED,
):
    chance_rng = (
        np.random.default_rng(seed)
    )
    chance_predictions = (
        chance_rng.integers(
            low=0,
            high=NUM_CLASSES,
            size=len(y_reference),
        )
    )
    majority_predictions = np.full(
        shape=len(y_reference),
        fill_value=majority_class_id,
        dtype=np.int32,
    )

    return {
        'Chance Baseline': (
            chance_predictions
        ),
        'Majority-Class Baseline': (
            majority_predictions
        ),
    }


print(
    'Training class probabilities:'
)
for class_name, probability in zip(
    CLASS_NAMES,
    class_probabilities,
):
    print(
        f'  {class_name}: '
        f'{probability:.4f}'
    )

print(
    'Majority class: '
    f'{CLASS_NAMES[majority_class_id]}'
)


# Step 20: Evaluate the held-out test fold exactly once

This is the final unbiased evaluation. No decision below this point
changes the model.


In [ ]:
test_evaluation = final_model.evaluate(
    embeddings_by_split['test'],
    y_test,
    batch_size=best_config[
        'batch_size'
    ],
    verbose=1,
    return_dict=True,
)
test_probabilities = (
    final_model.predict(
        embeddings_by_split['test'],
        batch_size=best_config[
            'batch_size'
        ],
        verbose=1,
    )
)
test_unadjusted_predictions = np.argmax(
    test_probabilities,
    axis=1,
)

test_adjusted_probabilities = (
    apply_left_turn_probability_multiplier(
        test_probabilities,
        selected_left_turn_multiplier,
    )
)
test_predictions = np.argmax(
    test_adjusted_probabilities,
    axis=1,
)

test_unadjusted_metrics = calculate_metrics(
    y_test,
    test_unadjusted_predictions,
    (
        'Kinetics-Pretrained MoViNet Head '
        '— Unadjusted'
    ),
)
test_metrics = calculate_metrics(
    y_test,
    test_predictions,
    (
        'Kinetics-Pretrained MoViNet Head '
        '— Adjusted'
    ),
)

print('Keras test evaluation:')
print(test_evaluation)
print(
    'Locked left-turn multiplier applied: '
    f'{selected_left_turn_multiplier:.2f}'
)
print('\n3D v3 test metrics:')
display(
    pd.DataFrame([
        test_unadjusted_metrics,
        test_metrics,
    ]).set_index('model')
)
print(
    '\n3D v3 adjusted test classification report:'
)
display(
    classification_report_df(
        y_test,
        test_predictions,
    )
)
plot_confusion_matrices(
    y_test,
    test_predictions,
    (
        'Selected Kinetics-Pretrained MoViNet Head '
        'Test — Adjusted'
    ),
)


# Step 21: Compare the transfer-learning model with both baselines

Macro F1 is the primary comparison because the three classes are
imbalanced and left-turn performance is especially important.


In [ ]:
comparison_records = []

for (
    baseline_name,
    baseline_prediction,
) in baseline_predictions(
    y_test,
    seed=SEED,
).items():
    comparison_records.append(
        calculate_metrics(
            y_test,
            baseline_prediction,
            baseline_name,
        )
    )

comparison_records.extend([
    test_unadjusted_metrics,
    test_metrics,
])
comparison_df = pd.DataFrame(
    comparison_records
).set_index('model')

print(
    'Final held-out test comparison:'
)
display(comparison_df.round(4))

benchmark_macro_f1 = 0.607
print(
    '\nHand-crafted Trend+Flow Random '
    'Forest macro-F1 benchmark: '
    f'{benchmark_macro_f1:.3f}'
)
print(
    'Unadjusted MoViNet macro-F1 difference: '
    f'{test_unadjusted_metrics["f1_macro"] - benchmark_macro_f1:+.4f}'
)
print(
    'Adjusted MoViNet macro-F1 difference: '
    f'{test_metrics["f1_macro"] - benchmark_macro_f1:+.4f}'
)

comparison_plot_df = (
    comparison_df[
        ['accuracy', 'f1_macro']
    ]
    .sort_values('f1_macro')
)
ax = comparison_plot_df.plot(
    kind='barh',
    figsize=(10, 5),
    xlim=(0, 1),
)
ax.axvline(
    benchmark_macro_f1,
    color='black',
    linestyle='--',
    linewidth=1.2,
    label=(
        'Trend+Flow RF '
        'macro-F1 benchmark'
    ),
)
ax.set_title(
    'Held-Out Test Accuracy and Macro F1'
)
ax.set_xlabel('Score')
ax.grid(
    axis='x',
    alpha=0.25,
)
ax.legend(
    loc='lower right'
)
plt.tight_layout()
plt.show()


# Step 22: Save `CNN_3D_v3.npy` and aligned class labels

The exported learned feature is the final custom head's
`dense_hidden` activation. It is derived from the cached Kinetics video
embedding and preserves the exact order:

The multiplier changes only the final decision rule; it does not alter the
learned `dense_hidden` feature vector. The selected multiplier is therefore
saved separately in a small JSON configuration file.

The exported rows preserve the exact order:

1. official training rows;
2. shared validation rows;
3. held-out test rows.


In [ ]:
FEATURES_DIR = (
    PROJECT_ROOT
    / 'data'
    / 'processed'
    / 'waymo_e2e'
    / 'features'
)
FEATURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CNN_FEATURE_PATH = (
    FEATURES_DIR / 'CNN_3D_v3.npy'
)
CNN_LABEL_PATH = (
    FEATURES_DIR
    / 'labels_CNN_3D_v3.npy'
)
CNN_DECISION_CONFIG_PATH = (
    FEATURES_DIR
    / 'CNN_3D_v3_decision_config.json'
)

cnn_feature_extractor = keras.Model(
    inputs=final_model.inputs,
    outputs=final_model.get_layer(
        'dense_hidden'
    ).output,
    name='cnn_3d_v3_feature_extractor',
)

cnn_feature_splits = []

for split_name in SPLIT_NAMES:
    print(
        f'Extracting {split_name} '
        'CNN 3D v3 features...'
    )

    split_features = (
        cnn_feature_extractor.predict(
            embeddings_by_split[
                split_name
            ],
            batch_size=max(
                32,
                best_config[
                    'batch_size'
                ],
            ),
            verbose=1,
        )
    ).astype(np.float32)

    cnn_feature_splits.append(
        split_features
    )

    print(
        f'  {split_name}: '
        f'{split_features.shape}'
    )

cnn_features = np.concatenate(
    cnn_feature_splits,
    axis=0,
)

cnn_labels = np.concatenate([
    split_label_dfs[
        split_name
    ]['class_label']
    .astype(str)
    .to_numpy()
    for split_name in SPLIT_NAMES
]).astype('<U20')

if (
    cnn_features.shape[0]
    != cnn_labels.shape[0]
):
    raise ValueError(
        f'Feature/label row mismatch: '
        f'{cnn_features.shape[0]} '
        'features and '
        f'{cnn_labels.shape[0]} '
        'labels.'
    )

if not np.isfinite(
    cnn_features
).all():
    raise ValueError(
        'CNN features contain NaN '
        'or infinite values.'
    )

np.save(
    CNN_FEATURE_PATH,
    cnn_features,
)
np.save(
    CNN_LABEL_PATH,
    cnn_labels,
)

decision_config = {
    'left_turn_probability_multiplier': (
        selected_left_turn_multiplier
    ),
    'left_turn_class_id': (
        LEFT_TURN_CLASS_ID
    ),
    'class_names': CLASS_NAMES,
    'selection_source': (
        'train-internal validation only'
    ),
    'best_head_config': best_config,
    'selected_head_epochs': (
        selected_head_epochs
    ),
}
with open(
    CNN_DECISION_CONFIG_PATH,
    'w',
    encoding='utf-8',
) as decision_config_file:
    json.dump(
        decision_config,
        decision_config_file,
        indent=2,
    )

print(
    '\nSaved CNN 3D v3 feature files:'
)
print(f'  {CNN_FEATURE_PATH}')
print(f'  {CNN_LABEL_PATH}')
print(f'  {CNN_DECISION_CONFIG_PATH}')
print(
    '\nCNN feature shape: '
    f'{cnn_features.shape}'
)
print(
    'CNN label shape: '
    f'{cnn_labels.shape}'
)

row_start = 0
for split_name in SPLIT_NAMES:
    row_end = (
        row_start
        + len(
            embeddings_by_split[
                split_name
            ]
        )
    )
    print(
        f'{split_name}: rows '
        f'{row_start:,} through '
        f'{row_end - 1:,}'
    )
    row_start = row_end

print('\nLabel distribution:')
print(
    pd.Series(cnn_labels)
    .value_counts()
    .reindex(CLASS_NAMES)
)


## Output summary

A successful run produces:

- reusable RGB clip caches in
  `data/processed/waymo_e2e/CNN_3D_v2/clip_cache/`;
- reusable frozen Kinetics MoViNet embeddings in
  `data/processed/waymo_e2e/CNN_3D_v2/embedding_cache/`;
- one prior-only example GIF per class in
  `data/processed/waymo_e2e/CNN_3D_v2/example_gifs/`;
- head-tuning tables, training curves, and left-turn multiplier curves
  based only on a train-internal validation fold;
- shared-validation and held-out-test metrics, classification reports,
  and count plus row-normalized confusion matrices;
- a baseline comparison against chance, majority class, and the
  `0.607` Trend+Flow Random Forest macro-F1 benchmark;
- `CNN_3D_v3.npy`, `labels_CNN_3D_v3.npy`, and the locked decision
  configuration under `data/processed/waymo_e2e/features/`.

Version 3 intentionally reuses the v2 clip and embedding caches. The
expensive video-backbone stage is skipped as long as the clip settings,
split fingerprints, and embedding files are unchanged.
